# Sampler Validation: Marginal Checks

For each validation target, we run the Boomerang and Sticky Boomerang,
resample uniformly in time, and overlay sample histograms against the
known marginal densities.

In [ ]:
import os
os.chdir('../..')

import numpy as np
import matplotlib.pyplot as plt

from benchmarks_august.targets.validation import gaussian_sanity_check, beta_binomial, neals_funnel, rosenbrock_banana
from benchmarks_august.samplers.factories import build_sampler
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path, resample_sticky_pdmp_path
from benchmarks_august.samplers.warmstart import warmup_reference

In [ ]:
# ── Shared settings ──────────────────────────────────────────────
N_SKELETON  = 20000
N_RESAMPLE  = 500000
BURNIN_FRAC = 0.1
refresh_rate = 1.0

def run_and_resample(sampler, target, sticky=False, warmup=True):
    """Warmup, preprocess, sample, and return time-uniform resamples."""
    if warmup:
        warmup_reference(sampler, n_rounds=3, n_pilot=500,
                         sticky=sticky, target=target)
    else:
        method = target.meta.get('preprocess_method', 'diagonal')
        if method == 'manual':
            sampler.preprocess(method='manual',
                               x_ref=target.x_ref,
                               Sigma_inv=target.Sigma_inv)
        else:
            sampler.preprocess(method='diagonal')
    
    sampler.reset(N=N_SKELETON)
    sampler.sample_auto(diagnostics=False)
    
    if sticky:
        _, samples = resample_sticky_pdmp_path(sampler, n_samples=N_RESAMPLE,
                                               burnin_frac=BURNIN_FRAC)
    else:
        _, samples = resample_pdmp_path(sampler, n_samples=N_RESAMPLE,
                                        burnin_frac=BURNIN_FRAC)
    return samples


def plot_marginals(target, samples_dict, figname=None):
    """Plot marginal histograms against true densities for each coordinate."""
    marginals = target.meta['marginal_grids']
    D = target.D
    n_samplers = len(samples_dict)
    
    fig, axes = plt.subplots(n_samplers, D, figsize=(3.5 * D, 3 * n_samplers),
                             squeeze=False)
    
    colors = ['steelblue', 'darkorange', 'seagreen', 'firebrick']
    
    for row, (label, samples) in enumerate(samples_dict.items()):
        for col in range(D):
            ax = axes[row, col]
            mg = marginals[col]
            
            ax.hist(samples[:, col], bins=120, density=True, alpha=0.5,
                    color=colors[row % len(colors)], label=label)
            ax.plot(mg['grid'], mg['pdf'], 'k-', lw=1.5, label='True')
            
            if row == 0:
                ax.set_title(mg['label'])
            if col == 0:
                ax.set_ylabel(label)
            if row == n_samplers - 1:
                ax.set_xlabel(mg['label'])
            ax.legend(fontsize=7, loc='upper right')
    
    fig.suptitle(target.name, fontsize=14, y=1.02)
    fig.tight_layout()
    if figname:
        fig.savefig(figname, dpi=150, bbox_inches='tight')
    plt.show()
    
    

## 0. Gaussian sanity check

In [ ]:
target_gauss = gaussian_sanity_check(D=5)

s_gauss = build_sampler('boomerang', target_gauss, N=N_SKELETON, refresh_rate=refresh_rate)
samp_gauss = run_and_resample(s_gauss, target_gauss, warmup=False)

s_gauss_pli = build_sampler('boomerang_pli', target_gauss, N=N_SKELETON, refresh_rate=refresh_rate)
samp_gauss_pli = run_and_resample(s_gauss_pli, target_gauss, warmup=False)

# Check: should be zero bounces
df = s_gauss.diagnostics_df
n_bounces = df[(df['event_type'] == 'bounce') & (df['accepted'] == True)].shape[0]
n_refresh = df[df['event_type'] == 'refresh'].shape[0]
wall = df['wall_seconds'].sum()
grad_evals = s_gauss.gradient_evals
df_pli = s_gauss_pli.diagnostics_df
n_bounces_pli = df_pli[(df_pli['event_type'] == 'bounce') & (df_pli['accepted'] == True)].shape[0]
n_refresh_pli = df_pli[df_pli['event_type'] == 'refresh'].shape[0]
wall_pli = df_pli['wall_seconds'].sum()
grad_evals_pli = s_gauss_pli.gradient_evals

print("--------- Boomerang ---------")
print(f"Accepted bounces: {n_bounces}  (expect 0)")
print(f"Refreshments:     {n_refresh}  (expect all skeleton points)")
print(f"Walltime:         {wall}")
print(f"Grad evals per skeleton point: {grad_evals / s_gauss.N:.1f}")
print("--------- Boomerang PLI ---------")
print(f"Accepted bounces: {n_bounces_pli}  (expect 0)")
print(f"Refreshments:     {n_refresh_pli}  (expect all skeleton points)")
print(f"Walltime:         {wall_pli}")
print(f"Grad evals per skeleton point: {grad_evals_pli / s_gauss_pli.N:.1f}")

plot_marginals(target_gauss, {'Boomerang': samp_gauss,
                              'Boomerang PLI': samp_gauss_pli})
               #,figname='validation_gaussian_refcheck.pdf')

## 1. Beta-Binomial (D=5)

In [ ]:
target_bb = beta_binomial(
    D=5,
    n_obs=np.array([30, 50, 20, 40, 60]),
    x_obs=np.array([20, 25, 5, 30, 35]),
    a_prior=2.0, b_prior=2.0,
)

# Boomerang
s_bb = build_sampler('boomerang', target_bb, N=N_SKELETON, refresh_rate=refresh_rate)
samp_bb = run_and_resample(s_bb, target_bb)

# Boomerang PLI
s_bb_pli = build_sampler('boomerang_pli', target_bb, N=N_SKELETON, refresh_rate=refresh_rate)
samp_bb_pli = run_and_resample(s_bb_pli, target_bb)

plot_marginals(target_bb, {
    'Boomerang': samp_bb,
    'Boomerang PLI': samp_bb_pli,
})#, figname='validation_beta_binomial.pdf')



In [ ]:
from Filippo_plotting.mcmc_plots import better_pairs

labels_bb = [target_bb.meta['marginal_grids'][i]['label'] for i in range(target_bb.D)]

fig, axes = better_pairs(samp_bb, resol=0.7, labels=labels_bb,
                         title='Boomerang beta-binomial')
plt.show()

fig, axes = better_pairs(samp_bb_pli, resol=0.7, labels=labels_bb,
                         title='Boomerang PLI beta-binomial')
plt.show()

## 2. Neal's Funnel (D=4)

In [ ]:
target_funnel = neals_funnel(D=4, sigma_v=3.0)

# Boomerang
s_funnel = build_sampler('boomerang', target_funnel, N=N_SKELETON)
samp_funnel = run_and_resample(s_funnel, target_funnel)

# Boomerang PLI
s_funnel_pli = build_sampler('boomerang_pli', target_funnel, N=N_SKELETON)
samp_funnel_pli = run_and_resample(s_funnel_pli, target_funnel)

plot_marginals(target_funnel, {
    'Boomerang': samp_funnel,
    'Boomerang PLI': samp_funnel_pli,
})#, figname='validation_neals_funnel.pdf')

## 3. Rosenbrock Banana (D=2)

In [ ]:
target_banana = rosenbrock_banana(a=1.0)

# Boomerang
s_banana = build_sampler('boomerang', target_banana, N=N_SKELETON)
samp_banana = run_and_resample(s_banana, target_banana)

# Boomerang PLI
s_banana_pli = build_sampler('boomerang_pli', target_banana, N=N_SKELETON)
samp_banana_pli = run_and_resample(s_banana_pli, target_banana)

plot_marginals(target_banana, {
    'Boomerang': samp_banana,
    'Boomerang PLI': samp_banana_pli,
})#, figname='validation_rosenbrock_banana.pdf')